In [0]:
from pyspark.sql.functions import avg, max, stddev, sum, when, col

In [0]:
silver_sales_path = "/Volumes/workspace/m5_project/silver/silver_sales"

silver_sales_df = spark.read.parquet(
    silver_sales_path
)

display(silver_sales_df.limit(10))

id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk
HOUSEHOLD_1_363_TX_3_validation,HOUSEHOLD_1_363,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_364_TX_3_validation,HOUSEHOLD_1_364,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_365_TX_3_validation,HOUSEHOLD_1_365,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_366_TX_3_validation,HOUSEHOLD_1_366,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_367_TX_3_validation,HOUSEHOLD_1_367,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_368_TX_3_validation,HOUSEHOLD_1_368,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_369_TX_3_validation,HOUSEHOLD_1_369,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_370_TX_3_validation,HOUSEHOLD_1_370,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,2,2011-01-29,11101
HOUSEHOLD_1_371_TX_3_validation,HOUSEHOLD_1_371,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_372_TX_3_validation,HOUSEHOLD_1_372,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,6,2011-01-29,11101


In [0]:
gold_inventory_df = silver_sales_df.groupBy(
    "item_id",
    "store_id",
    "cat_id",
    "dept_id",
    "state_id"
).agg(
    avg("sales").alias("avg_daily_sales"),

    max("sales").alias("max_daily_sales"),

    stddev("sales").alias("sales_volatility"),

    sum(
        when(
            col("sales") == 0,
            1
        ).otherwise(0)
    ).alias("zero_sales_days")
)

In [0]:
gold_inventory_df = (
    gold_inventory_df
    .withColumn("category", col("cat_id"))
    .withColumn("department", col("dept_id"))
)

In [0]:
# gold_inventory_df = gold_inventory_df.withColumn(
#     "risk_level",
#     when(
#         (col("avg_daily_sales") > 5) &
#         (col("sales_volatility") > 10),
#         "HIGH"
#     ).when(
#         (col("avg_daily_sales") > 2),
#         "MEDIUM"
#     ).otherwise(
#         "LOW"
#     )
# )

# display(gold_inventory_df.limit(20))


volatility_threshold = gold_inventory_df.approxQuantile(
    "sales_volatility",
    [0.75],
    0.01
)[0]

zero_sales_threshold = gold_inventory_df.approxQuantile(
    "zero_sales_days",
    [0.75],
    0.01
)[0]

low_sales_threshold = gold_inventory_df.approxQuantile(
    "avg_daily_sales",
    [0.25],
    0.01
)[0]

In [0]:
# =====================================
# DATA-DRIVEN RISK CLASSIFICATION
# =====================================

print("Volatility Threshold (75th percentile):", volatility_threshold)
print("Zero Sales Threshold (75th percentile):", zero_sales_threshold)
print("Low Sales Threshold (25th percentile):", low_sales_threshold)


gold_inventory_df = gold_inventory_df.withColumn(
    "risk_level",

    # HIGH RISK
    when(
        (
            (
                (col("sales_volatility") > volatility_threshold)
                &
                (col("zero_sales_days") > zero_sales_threshold)
            )
            |
            (
                (col("avg_daily_sales") < low_sales_threshold)
                &
                (col("zero_sales_days") > zero_sales_threshold)
            )
        ),
        "HIGH"
    )

    # MEDIUM RISK
    .when(
        (
            (col("sales_volatility") > volatility_threshold * 0.7)
            |
            (col("zero_sales_days") > zero_sales_threshold * 0.7)
            |
            (col("avg_daily_sales") < low_sales_threshold * 1.5)
        ),
        "MEDIUM"
    )

    # LOW RISK
    .otherwise(
        "LOW"
    )
)


# =====================================
# VALIDATION
# =====================================

display(
    gold_inventory_df.select(
        "item_id",
        "category",
        "department",
        "store_id",
        "state_id",
        "avg_daily_sales",
        "sales_volatility",
        "zero_sales_days",
        "risk_level"
    ).limit(20)
)

Volatility Threshold (75th percentile): 1.5370850993905807
Zero Sales Threshold (75th percentile): 1657.0
Low Sales Threshold (25th percentile): 0.18714061683220073


item_id,category,department,store_id,state_id,avg_daily_sales,sales_volatility,zero_sales_days,risk_level
HOUSEHOLD_2_006,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.07631991636173549,0.2845914749985213,1777,HIGH
HOUSEHOLD_2_050,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.14584422373235756,0.4002545356323709,1665,HIGH
HOUSEHOLD_2_056,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.09984317825405123,0.332929460401573,1741,HIGH
HOUSEHOLD_2_122,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.20543648719289076,0.48849924975916653,1583,MEDIUM
HOUSEHOLD_2_127,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.11029796131730267,0.3892697340109698,1744,HIGH
HOUSEHOLD_2_232,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.29796131730266595,0.5867086271286778,1458,MEDIUM
HOUSEHOLD_2_285,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.14636696288552012,0.4135627241109713,1667,HIGH
HOUSEHOLD_2_462,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.11395713538944068,0.35069964774937956,1715,HIGH
HOUSEHOLD_2_468,HOUSEHOLD,HOUSEHOLD_2,TX_3,TX,0.2624150548876111,0.525633530108258,1485,MEDIUM
FOODS_1_107,FOODS,FOODS_1,TX_3,TX,1.0862519602718244,2.2550232660011877,1315,MEDIUM


In [0]:
gold_inventory_path = "/Volumes/workspace/m5_project/gold/gold_inventory_risk"

gold_inventory_df.write.mode("overwrite").parquet(
    gold_inventory_path
)

print("Gold Inventory Risk written successfully")

Gold Inventory Risk written successfully


In [0]:
gold_check_df = spark.read.parquet(
    gold_inventory_path
)

display(gold_check_df.limit(20))

item_id,store_id,cat_id,dept_id,state_id,avg_daily_sales,max_daily_sales,sales_volatility,zero_sales_days,category,department,risk_level
HOUSEHOLD_2_006,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.07631991636173549,2,0.2845914749985213,1777,HOUSEHOLD,HOUSEHOLD_2,HIGH
HOUSEHOLD_2_050,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.14584422373235756,3,0.4002545356323709,1665,HOUSEHOLD,HOUSEHOLD_2,HIGH
HOUSEHOLD_2_056,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.09984317825405123,3,0.332929460401573,1741,HOUSEHOLD,HOUSEHOLD_2,HIGH
HOUSEHOLD_2_122,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.20543648719289076,3,0.48849924975916653,1583,HOUSEHOLD,HOUSEHOLD_2,MEDIUM
HOUSEHOLD_2_127,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.11029796131730267,4,0.3892697340109698,1744,HOUSEHOLD,HOUSEHOLD_2,HIGH
HOUSEHOLD_2_232,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.29796131730266595,3,0.5867086271286778,1458,HOUSEHOLD,HOUSEHOLD_2,MEDIUM
HOUSEHOLD_2_285,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.14636696288552012,6,0.4135627241109713,1667,HOUSEHOLD,HOUSEHOLD_2,HIGH
HOUSEHOLD_2_462,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.11395713538944068,3,0.35069964774937956,1715,HOUSEHOLD,HOUSEHOLD_2,HIGH
HOUSEHOLD_2_468,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.2624150548876111,3,0.525633530108258,1485,HOUSEHOLD,HOUSEHOLD_2,MEDIUM
FOODS_1_107,TX_3,FOODS,FOODS_1,TX,1.0862519602718244,22,2.2550232660011877,1315,FOODS,FOODS_1,MEDIUM


In [0]:
gold_inventory_export_path = "/Volumes/workspace/m5_project/gold/exports/gold_inventory_risk_csv"

gold_inventory_df.write.mode("overwrite") \
    .option("header", True) \
    .csv(gold_inventory_export_path)

print("Inventory CSV export complete")

Inventory CSV export complete
